## Testing Gemini API

In [32]:
import os
import google.generativeai as genai
from dotenv import load_dotenv
load_dotenv(r"D:\Workspace\workspace\Analytics\apps\AI_Data_Engineering\ai\.env", override=True)

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-3.6-flash")
response = model.generate_content("Say hello, mention which model you are using.")
print(response.text)

Hello! I am Gemini, a large language model created by Google. How can I help you today?


In [33]:
load_dotenv(r"D:\Workspace\workspace\Analytics\apps\AI_Data_Engineering\ai\.env", override=True)
print("SNOWFLAKE_ACCOUNT =", os.getenv("SNOWFLAKE_ACCOUNT"))

SNOWFLAKE_ACCOUNT = CVDTUSX-BG23471


In [34]:
import os
import json
import re
import snowflake.connector
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv(r"D:\Workspace\workspace\Analytics\apps\AI_Data_Engineering\ai\.env", override=True)
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

# Gemini model and review labels.
MODEL = "gemini-3.6-flash"
SAMPLE_N = 5
TOPICS = ["food quality", "delivery", "pricing", "service", "packaging", "other"]
PROMPT = f"You classify customer reviews for a food delivery app. Return JSON with sentiment_label, sentiment_score, topic ({TOPICS}), and key_issue (or null)."
model = genai.GenerativeModel(MODEL)


# Connect to Snowflake using environment variables.
def get_connection():
    return snowflake.connector.connect(
        user=os.getenv("SNOWFLAKE_USER"),
        password=os.getenv("SNOWFLAKE_PASSWORD"),
        account=os.getenv("SNOWFLAKE_ACCOUNT"),
        warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
        database=os.getenv("SNOWFLAKE_DATABASE"),
        schema=os.getenv("SNOWFLAKE_SCHEMA"),
    )


# Create the target table once if it does not already exist.
def create_output_table(cursor):
    cursor.execute("CREATE SCHEMA IF NOT EXISTS FOOD_DELIVERY.AI")
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS FOOD_DELIVERY.AI.REVIEW_ENRICHED (
            REVIEW_ID STRING,
            SENTIMENT_LABEL STRING,
            SENTIMENT_SCORE FLOAT,
            TOPIC STRING,
            KEY_ISSUE STRING,
            MODEL STRING,
            ENRICHED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
        )
    """)


# Pull reviews that have not been enriched yet.
def get_reviews_to_enrich(cursor):
    cursor.execute(f"""
        SELECT REVIEW_ID, COMMENT
        FROM FOOD_DELIVERY.RAW.REVIEWS
        WHERE REVIEW_ID NOT IN (
            SELECT REVIEW_ID FROM FOOD_DELIVERY.AI.REVIEW_ENRICHED
        )
        LIMIT {SAMPLE_N}
    """)
    return cursor.fetchall()


# Extract the JSON object from Gemini output.
def parse_json_response(text):
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in model output: {text}")
    return json.loads(cleaned[start:end + 1])


# Ask Gemini to classify one review and return parsed JSON.
def classify_review(comment):
    try:
        response = model.generate_content(
            f"{PROMPT}\n\nReview: {comment}",
            generation_config={"temperature": 0},
        )
        return parse_json_response(response.text)
    except Exception as e:
        message = str(e)
        if "429" in message or "quota" in message.lower():
            print("Gemini quota reached. Stop this batch and try again after quota resets or on a paid plan.")
            return None
        raise


# Insert all enriched rows into Snowflake in one batch.
def save_results(cursor, results):
    print(f"Saving {len(results)} enriched reviews to Snowflake...")
    if not results:
        return
    cursor.executemany(
        """
        INSERT INTO FOOD_DELIVERY.AI.REVIEW_ENRICHED
            (review_id, sentiment_label, sentiment_score, topic, key_issue, model)
        VALUES (%s, %s, %s, %s, %s, %s)
        """,
        results,
    )


# End-to-end flow: fetch, classify, and save.
def main():
    conn = get_connection()
    cursor = conn.cursor() # the object you use to run SQL commands
    create_output_table(cursor)
    reviews = get_reviews_to_enrich(cursor)

    if len(reviews) == 0:
        print("No new reviews to enrich.")
        return

    print(f"Enriching {len(reviews)} reviews...")

    # Classify each review and collect rows for a single batch insert.
    results = []
    for review_id, comment in reviews:
        print(f"Classifying review {review_id}: {comment}")
        labels = classify_review(comment)
        if labels is None:
            break
        print(f"Labels for review {review_id}: {labels}")
        results.append((
                review_id,
                labels["sentiment_label"],
                labels["sentiment_score"],
                labels["topic"],
                labels["key_issue"],
                MODEL,
            ))

    # Save all successfully classified reviews together.
    save_results(cursor, results)
    if results:
        print(f"Saved {len(results)} enriched reviews to Snowflake.")
        conn.commit()
    else:
        print("No reviews were enriched because Gemini quota was exhausted.")
    cursor.close()
    conn.close()


if __name__ == "__main__":
    main()

Enriching 5 reviews...
Classifying review 2: Far too expensive for what you get.
Labels for review 2: {'sentiment_label': 'negative', 'sentiment_score': 0.12, 'topic': 'pricing', 'key_issue': 'High prices relative to value'}
Classifying review 3: An average experience overall.
Labels for review 3: {'sentiment_label': 'neutral', 'sentiment_score': 0.5, 'topic': 'other', 'key_issue': None}
Classifying review 4: The delivery partner was polite and helpful.
Labels for review 4: {'sentiment_label': 'positive', 'sentiment_score': 0.95, 'topic': 'delivery', 'key_issue': None}
Classifying review 5: Great eco-friendly packaging.
Labels for review 5: {'sentiment_label': 'positive', 'sentiment_score': 0.95, 'topic': 'packaging', 'key_issue': None}
Classifying review 6: Gravy spilled all over the bag.
Labels for review 6: {'sentiment_label': 'negative', 'sentiment_score': -0.9, 'topic': 'packaging', 'key_issue': 'Gravy spilled inside the bag'}
Saving 5 enriched reviews to Snowflake...
Saved 5 enri